# Install Dependencies

In [ ]:
!pip install -q transformers==4.45.2
!pip install -q tokenizers==0.20.1
!pip install -q sentencepiece==0.2.0
!pip install datasets

In [ ]:
!pip install -q arabert

# Load dataset

In [ ]:
from datasets import load_dataset

data = load_dataset(
    "MBZUAI/ArabicMMLU",
    "Physics (High School)"
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [ ]:
import pandas as pd

dev = data["dev"]
test = data["test"]

dev_df = dev.to_pandas()
test_df = test.to_pandas()

df = pd.concat([dev_df, test_df])
df.columns

Index(['ID', 'Source', 'Country', 'Group', 'Subject', 'Level', 'Question',
       'Context', 'Answer Key', 'Option 1', 'Option 2', 'Option 3', 'Option 4',
       'Option 5', 'is_few_shot'],
      dtype='object')

In [ ]:
df.dropna(inplace = True, subset = 'Question')

In [ ]:
df.shape[0]

258

In [ ]:
test_samples = pd.read_excel('sample_physics.xlsx')
test_samples.head(1)

,ID,Source,Country,Group,Subject,Level,Question,Context,Answer Key,Option 1,Option 2,Option 3,Option 4,Option 5,is_few_shot
0,5148,https://drive.google.com/drive/folders/14FKkR6...,Palestine,STEM,Physics,High,مقاومتان مقدار كل منهما م إذا وصلتا على التو...,NaN,A,0.5 م,م,2 م,1.5 م,NaN,0


In [ ]:
test_samples.shape[0]

200

In [ ]:
data_filtered = df[~df["Question"].isin(test_samples["Question"])]
data_filtered .shape[0]

58

In [ ]:
data_filtered.reset_index(drop = True, inplace = True)

# Preprocess

In [ ]:
option_columns = [
    (":A", "Option 1"),
    ("B:", "Option 2"),
    ("C:", "Option 3"),
    ("D:", "Option 4"),
    ("E:", "Option 5"),
]

choices = []
for i in range(len(data_filtered)):
    options = []
    for letter, column in option_columns:
        value = data_filtered[column].iloc[i]

        options.append(f"{letter}: {value}")
    choices.append(
        ", ".join(options)
        )

data_filtered['Options'] = choices
data_filtered.head(1)

/tmp/ipykernel_8989/2868611666.py:20: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  data_filtered['Options'] = choices


,ID,Source,Country,Group,Subject,Level,Question,Context,Answer Key,Option 1,Option 2,Option 3,Option 4,Option 5,is_few_shot,Options
0,4976,exams-28a8ea10-7722-11ea-9116-54bef70b159e,None,STEM,Physics,High,أ هوب لٍ لا نٍطبق علي قوى الفعل و رد الفعل ؟,NaN,C,قىة الفعل و قىة رد الفعل تؤثران ف جسميه مختلفيه,قىة الفعل تسبوي قىة رد الفعل,قىة الفعل و قىة رد الفعل تؤثران ف وفس الجسم,قىة الفعل تعبكس قىة رد الفعل,NaN,1,:A: قىة الفعل و قىة رد الفعل تؤثران ف جسميه مخ...


In [ ]:
def build_prompt(data):
    articles = []

    for questions, choices, answer in zip(data["Question"], data["Options"], data["Answer Key"]):
        inputs = {"question": questions, "choices": choices, "answer": answer}
        articles.append(inputs)

    return articles

In [ ]:
data = build_prompt(data_filtered)

# Create a Dataframe
data = pd.DataFrame(data)

In [ ]:
data.head(1)

,question,choices,answer
0,أ هوب لٍ لا نٍطبق علي قوى الفعل و رد الفعل ؟,:A: قىة الفعل و قىة رد الفعل تؤثران ف جسميه مخ...,C


In [ ]:
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5ForConditionalGeneration,
)

# Configuration
CHECKPOINT = "UBC-NLP/AraT5v2-base-1024"

# Convert your pandas DataFrame to a Hugging Face Dataset
dataset = Dataset.from_pandas(data)

# Split dataset into 80% train and 20% validation
dataset_split = dataset.train_test_split(test_size=0.2, seed=42)
train_dataset = dataset_split["train"]
eval_dataset = dataset_split["test"]

# Load Tokenizer and Model
tokenizer = AutoTokenizer.from_pretrained(CHECKPOINT)
model = T5ForConditionalGeneration.from_pretrained(CHECKPOINT)

# Preprocessing

In [ ]:
# Data Preprocessing Function
def preprocess_function(examples):
    inputs = []
    for q, c in zip(examples["question"], examples["choices"]):
        prompt = f"حل السؤال التالي متعدد الخيارات. سؤال: {q} خيارات: {c}"
        inputs.append(prompt)

    targets = [str(a).strip() for a in examples["answer"]]

    # Tokenize inputs and labels
    model_inputs = tokenizer(inputs, max_length=512, truncation=True, padding=False)
    labels = tokenizer(text_target=targets, max_length=64, truncation=True, padding=False)

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs


# Tokenize the Split Datasets
tokenized_train = train_dataset.map(
    preprocess_function, batched=True, remove_columns=train_dataset.column_names
)
tokenized_eval = eval_dataset.map(
    preprocess_function, batched=True, remove_columns=eval_dataset.column_names
)

# Data Collator for Dynamic Padding
data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True
)

Map:   0%|          | 0/46 [00:00<?, ? examples/s]

Map:   0%|          | 0/12 [00:00<?, ? examples/s]

# Define Arguments

In [ ]:
# Define Training Arguments
training_args = Seq2SeqTrainingArguments(
    output_dir="./arat5_mcq_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=5e-5,
    per_device_train_batch_size=4,  # Kept small due to large 1024 context model size
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,  # Simulates effective batch size of 8
    weight_decay=0.01,
    num_train_epochs=15,
    predict_with_generate=True,
    fp16=True,  # Set to False if your GPU doesn't support mixed precision
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none",  # Prevents wandb login popups
)

# Initialize Seq2SeqTrainer
trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_eval,
    tokenizer=tokenizer,
    data_collator=data_collator,
)


# Train

In [ ]:
# Start Training
trainer.train()

# Save the Final Model and Tokenizer
model.save_pretrained("./best_arat5_mcq_model")
tokenizer.save_pretrained("./best_arat5_mcq_model")


Epoch,Training Loss,Validation Loss
1,No log,14.860026
2,19.820100,8.539216
3,19.820100,7.980627
4,16.699300,7.425956
5,12.207700,6.625113
6,12.207700,5.722029
7,9.067300,4.468109
8,9.067300,2.239098
9,7.665200,1.337485
10,5.111600,0.943523


There were missing keys in the checkpoint model loaded: ['encoder.embed_tokens.weight', 'decoder.embed_tokens.weight'].


('./best_arat5_mcq_model/tokenizer_config.json',
 './best_arat5_mcq_model/special_tokens_map.json',
 './best_arat5_mcq_model/spiece.model',
 './best_arat5_mcq_model/added_tokens.json',
 './best_arat5_mcq_model/tokenizer.json')

# Evaluate

In [ ]:
choices = []
for i in range(len(test_samples)):
    options = []
    for letter, column in option_columns:
        value = test_samples[column].iloc[i]

        options.append(f"{letter}: {value}")
    choices.append(
        "\, ".join(options)
        )

test_samples['Options'] = choices
test_samples = build_prompt(test_samples)

# Create a Dataframe
test_samples = pd.DataFrame(test_samples)
test_samples.head(1)

<>:9: SyntaxWarning: invalid escape sequence '\,'
<>:9: SyntaxWarning: invalid escape sequence '\,'
/tmp/ipykernel_8989/727638911.py:9: SyntaxWarning: invalid escape sequence '\,'
  "\, ".join(options)


,question,choices,answer
0,مقاومتان مقدار كل منهما م إذا وصلتا على التو...,":A: 0.5 م\, B:: م\, C:: 2 م\, D:: 1.5 م\, E:: nan",A


In [ ]:
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
    T5ForConditionalGeneration,
)

MODEL_PATH = "./best_arat5_mcq_model"

tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
model = T5ForConditionalGeneration.from_pretrained(MODEL_PATH)

test_dataset = Dataset.from_pandas(test_samples)

def preprocess_test_function(examples):
  inputs = [
      f"question: {q} choices: {c}"
      for q, c in zip(examples["question"], examples["choices"])
  ]
  targets = [str(a) for a in examples["answer"]]

  model_inputs = tokenizer(
      inputs, max_length=512, truncation=True, padding=False
  )
  labels = tokenizer(
      text_target=targets, max_length=128, truncation=True, padding=False
  )

  model_inputs["labels"] = labels["input_ids"]
  return model_inputs

# Tokenize the test dataset
tokenized_test = test_dataset.map(
    preprocess_test_function,
    batched=True,
    remove_columns=test_dataset.column_names,
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./test_predictions",
    per_device_eval_batch_size=8,
    predict_with_generate=True,
    generation_max_length=128,
    report_to="none",
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer, model=model, padding=True
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    tokenizer=tokenizer,
    data_collator=data_collator,
)

print("Running batch prediction on test dataframe...")
predictions = trainer.predict(tokenized_test)

raw_predictions = predictions.predictions
raw_labels = predictions.label_ids

raw_predictions = np.where(
    raw_predictions != -100, raw_predictions, tokenizer.pad_token_id
)
raw_labels = np.where(raw_labels != -100, raw_labels, tokenizer.pad_token_id)

decoded_preds = tokenizer.batch_decode(
    raw_predictions, skip_special_tokens=True
)
decoded_labels = tokenizer.batch_decode(raw_labels, skip_special_tokens=True)

test_samples["predicted_answer"] = [p.strip() for p in decoded_preds]

test_samples.head()

Map:   0%|          | 0/200 [00:00<?, ? examples/s]

Running batch prediction on test dataframe...


,question,choices,answer,predicted_answer
0,مقاومتان مقدار كل منهما م إذا وصلتا على التو...,":A: 0.5 م\, B:: م\, C:: 2 م\, D:: 1.5 م\, E:: nan",A,D
1,ما هو مدى الموجات الصوت ةٌ الت تٌمكن الإنسان ا...,":A: أكبر من Hz 20000\, B:: من Hz 10 إلى Hz 100...",D,D
2,جسم كتلته 5 كغم وكمية تحركه 15 كغم.م/ث ، فإن ...,":A: 1.7 نيوتن\, B:: 0.35 نيوتن\, C:: 5.1 نيوتن...",A,D
3,الفرق في درجات الحرارة الالزمة لمادة معاملها ا...,":A: 20 كلفن\, B:: 5 كلفن\, C:: 12 كلفن\, D:: 1...",D,D
4,كمية التحرك للنظام الذي يتكون من كرتين متماثل...,":A: 2 ك ع\, B:: صفر\, C:: ك ع\, D:: 0.5 ك ع\, ...",B,B


In [ ]:
test_samples["predicted_answer"].value_counts()

,count
predicted_answer,
D,127
B,73


In [ ]:
test_samples.to_excel('QA Physics by AraT5.xlsx', index = False)
test_samples.head(2)

,question,choices,answer,predicted_answer
0,مقاومتان مقدار كل منهما م إذا وصلتا على التو...,":A: 0.5 م\, B:: م\, C:: 2 م\, D:: 1.5 م\, E:: nan",A,D
1,ما هو مدى الموجات الصوت ةٌ الت تٌمكن الإنسان ا...,":A: أكبر من Hz 20000\, B:: من Hz 10 إلى Hz 100...",D,D


In [ ]:
from sklearn.metrics import classification_report

print(classification_report(test_samples['answer'],
                            test_samples['predicted_answer'],
                            digits =4 ))

              precision    recall  f1-score   support

           A     0.0000    0.0000    0.0000        49
           B     0.2740    0.3448    0.3053        58
           C     0.0000    0.0000    0.0000        42
           D     0.2520    0.6275    0.3596        51

    accuracy                         0.2600       200
   macro avg     0.1315    0.2431    0.1662       200
weighted avg     0.1437    0.2600    0.1802       200



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
